# Ingest

load pdf, chunk documents, convert into embeddings, store in ChromaDB

In [ ]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
VECTOR_DB_PATH = "../irc_vectordb"

### Load PDF

In [ ]:
# Load pdf - took ~25 sec for me the first time I ran this cell
# Im on a Macbook Pro M1 8GB 2020 for reference

IRC_path = '../Internal Revenue Code/IRC.pdf'
loader = PyMuPDFLoader(IRC_path)
documents = loader.load()

In [ ]:
documents[100]

### Chunk Documents

In [ ]:
# Chunk documents - for reference: chunk_size = 1000 & chunk_overlap = 200 yielded ~34k chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

print(f"Number of Chunks: {len(chunks)} chunks")

### Create embeddings and store

In [ ]:
# Create embeddings and store - took ~6 min

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

if os.path.exists(VECTOR_DB_PATH):
    Chroma(persist_directory=VECTOR_DB_PATH, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=VECTOR_DB_PATH)

In [ ]:
# Lets see the vectors

collection = vectorstore._collection
sample = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample)

print(f"Example Vector: {sample[:10]}") # seeing only 10 of numbers out of the 384
print(f"There are a total of {collection.count()} vectors with a dimension of {dimensions}")

At this point I moved to `answer.ipynd` but realized that the retriever was not retrieving relevent documents. 

I will now try a different strategy.

### Use XML format instead of PDF

instead of using the xml file as is, I will clean it to get rid of the amendments and other information that is not needed

Using AI to help clean the XML file since its a huge document


### Load XML file

In [107]:
import re
import json
import sys
import xml.etree.ElementTree as ET
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.schema import Document

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
IRC_PATH = "../Internal Revenue Code/IRC xml.xml"
VECTOR_DB_PATH = "../irc_xml_vectordb"

In [108]:

# ── Namespace ─────────────────────────────────────────────────────────────────
NS = "{http://xml.house.gov/schemas/uslm/1.0}"

# ── Tags to skip entirely (including all their children) ─────────────────────
SKIP_TAGS = {
    "notes", "note", "sourceCredit",
    "toc", "layout",
    "ref", "date", "meta",
}


In [150]:
def t(tag):
    """Strip namespace from a tag."""
    return tag.replace(NS, "")


def build_parent_map(root):
    """Build a child → parent mapping for the entire tree."""
    parent_map = {}
    for parent in root.iter():
        for child in parent:
            parent_map[child] = parent
    return parent_map


def get_ancestor_labels(section, parent_map):
    """
    Walk up the tree from a section element and collect
    subtitle, chapter, subchapter, part, subpart labels.
    """
    ancestors = {
        "subtitle":   "",
        "chapter":    "",
        "subchapter": "",
        "part":       "",
        "subpart":    "",
    }

    el = section
    while el is not None:
        el = parent_map.get(el)
        if el is None:
            break
        tag = t(el.tag)
        if tag in ancestors and not ancestors[tag]:
            # Get num + heading for this ancestor
            num_el     = el.find(f"{NS}num")
            heading_el = el.find(f"{NS}heading")
            num     = (num_el.text     or "").strip() if num_el     is not None else ""
            heading = (heading_el.text or "").strip() if heading_el is not None else ""
            ancestors[tag] = f"{num} {heading}".strip()

    return ancestors


def extract_legal_text(element):
    """
    Recursively extract clean legal text, skipping all noise tags.
    Preserves reading order including tail text.
    """
    tag = t(element.tag)

    if tag in SKIP_TAGS:
        return []

    texts = []

    if element.text and element.text.strip():
        texts.append(element.text.strip())

    for child in element:
        child_tag = t(child.tag)
        if child_tag not in SKIP_TAGS:
            texts.extend(extract_legal_text(child))
        if child.tail and child.tail.strip():
            if child_tag not in SKIP_TAGS:
                texts.append(child.tail.strip())

    return texts


def parse_and_clean(xml_path):
    print(f"Parsing {xml_path} ...")
    tree = ET.parse(xml_path)
    root = tree.getroot()

    main  = root.find(f"{NS}main")
    if main is None:
        print("ERROR: Could not find <main> element.")
        sys.exit(1)

    # Build parent map once — used for ancestor lookups
    print("  Building parent map ...")
    parent_map = build_parent_map(root)

    documents = []
    skipped   = 0

    # Iterate ALL section tags in document order
    all_sections = list(root.iter(f"{NS}section"))
    print(f"  Found {len(all_sections)} total <section> tags")

    for section in all_sections:
        identifier = section.get("identifier", "")

        # Only process real IRC sections (/us/usc/t26/sXXX)
        if "/s" not in identifier:
            skipped += 1
            continue

        section_num = identifier.split("/s")[-1]

        # ── Heading ───────────────────────────────────────────────────
        heading_el = section.find(f"{NS}heading")
        heading    = ""
        if heading_el is not None and heading_el.text:
            heading = heading_el.text.strip()

        # ── Clean legal text ──────────────────────────────────────────
        legal_parts = extract_legal_text(section)
        legal_text  = " ".join(legal_parts).strip()
        legal_text  = re.sub(r'\s+', ' ', legal_text)

        if len(legal_text) < 50:
            skipped += 1
            continue

        # ── Ancestor context ──────────────────────────────────────────
        ancestors = get_ancestor_labels(section, parent_map)

        # ── Bake section ID into text for retrieval ───────────────────

        documents.append(Document(
            page_content=legal_text,
            metadata={
                "section":    section_num,
                "heading":    heading,
                "identifier": identifier,
                "subtitle":   ancestors["subtitle"],
                "chapter":    ancestors["chapter"],
                "subchapter": ancestors["subchapter"],
                "part":       ancestors["part"],
                "subpart":    ancestors["subpart"],
                "source":     "IRC",
            }
        ))

    print(f"  Parsed  : {len(documents)} sections")
    print(f"  Skipped : {skipped} (no valid identifier or too short)")
    return documents


def chunk_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=4000,
        chunk_overlap=400,
    )
    chunks = splitter.split_documents(documents)
    
    # Prepend section identifier to EVERY chunk so BM25 can always find it
    for chunk in chunks:
        section = chunk.metadata.get("section", "")
        heading = chunk.metadata.get("heading", "")
        prefix  = f"§ {section} {heading}\n"
        
        # Only add prefix if it's not already there (first chunk already has it)
        if not chunk.page_content.startswith(f"§ {section}"):
            chunk.page_content = prefix + chunk.page_content
    
    print(f"  Chunked : {len(chunks)} chunks from {len(documents)} sections")
    return chunks


def save_chunks(chunks):
    """Save chunks to disk and save in vectordb"""

    output_path = "../Internal Revenue Code/irc_chunks.json"
    data = [
        {"text": c.page_content, "metadata": c.metadata}
        for c in chunks
    ]
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
        
    print(f"Saved {len(data)} chunks to {output_path}")

    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

    if os.path.exists(VECTOR_DB_PATH):
        Chroma(persist_directory=VECTOR_DB_PATH, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=VECTOR_DB_PATH)

    return vectorstore

In [152]:

documents = parse_and_clean(IRC_PATH)

if not documents:
    print("ERROR: No documents parsed.")
    sys.exit(1)

chunks = chunk_documents(documents)
vectorstore = save_chunks(chunks)


Parsing ../Internal Revenue Code/IRC xml.xml ...
  Building parent map ...
  Found 2275 total <section> tags
  Parsed  : 1898 sections
  Skipped : 377 (no valid identifier or too short)
  Chunked : 3955 chunks from 1898 sections
Saved 3955 chunks to ../Internal Revenue Code/irc_chunks.json


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6060.20it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [122]:
# Preview first section
print("\n── Sample Document ──────────────────────────────────────")
print(f"Total Documents: {len(documents)}")
s = documents[0]
print(f"  Section    : {s.metadata['section']}")
print(f"  Heading    : {s.metadata['heading']}")
print(f"  Subtitle   : {s.metadata['subtitle']}")
print(f"  Chapter    : {s.metadata['chapter']}")
print(f"  Subchapter : {s.metadata['subchapter']}")
print(f"  Part       : {s.metadata['part']}")
print(f"  Subpart    : {s.metadata['subpart']}")
print(f"  Text       : {s.page_content}")
print("──────────────────────────────────────────────────────")
print("\n── Sample chunk ──────────────────────────────────────")
print(f"Total Chunks: {len(chunks)}")
print(f"  Chunk: {chunks[1]}")
print("──────────────────────────────────────────────────────")


── Sample Document ──────────────────────────────────────
Total Documents: 1898
  Section    : 1
  Heading    : Tax imposed
  Subtitle   : Subtitle A— Income Taxes
  Chapter    : CHAPTER 1— NORMAL TAXES AND SURTAXES
  Subchapter : Subchapter A— Determination of Tax Liability
  Part       : PART I— TAX ON INDIVIDUALS
  Subpart    : 
  Text       : § 1. Tax imposed (a) Married individuals filing joint returns and surviving spouses There is hereby imposed on the taxable income of— (1) every married individual (as defined in section 7703) who makes a single return jointly with his spouse under section 6013, and (2) every surviving spouse (as defined in section 2(a)), a tax determined in accordance with the following table: If taxable income is: The tax is: Not over $36,900 15% of taxable income. Over $36,900 but not over $89,150 $5,535, plus 28% of the excess over $36,900. Over $89,150 but not over $140,000 $20,165, plus 31% of the excess over $89,150. Over $140,000 but not over $250,000 

In [119]:
# Lets see the vectors

collection = vectorstore._collection
sample = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample)

print(f"Example Vector: {sample[:10]}") # seeing only 10 of numbers out of the 384
print(f"There are a total of {collection.count()} vectors with a dimension of {dimensions}")

Example Vector: [-0.02492972  0.03132092  0.00998969 -0.07083808  0.03674274  0.02383781
  0.03647346  0.01926962 -0.03504835  0.07317344]
There are a total of 4011 vectors with a dimension of 384


### Checking length of documents and chunks
Trying to see if having bigger chunks makes more sense so no information is cut off and the question can be answered properly.

In [153]:

lengths = [len(d.page_content) for d in documents]

print(f"Total sections      : {len(lengths)}")
print(f"Average length      : {sum(lengths)//len(lengths)} chars")
print(f"Shortest section    : {min(lengths)} chars")
print(f"Longest section     : {max(lengths)} chars")
print(f"Sections over 4000  : {sum(1 for l in lengths if l > 4000)}")
print(f"Sections over 8000  : {sum(1 for l in lengths if l > 8000)}")
print(f"Sections over 16000 : {sum(1 for l in lengths if l > 16000)}")

Total sections      : 1898
Average length      : 5544 chars
Shortest section    : 72 chars
Longest section     : 153826 chars
Sections over 4000  : 682
Sections over 8000  : 346
Sections over 16000 : 143


In [154]:
# Length of chunks
lengths = [len(d.page_content) for d in chunks]

print(f"Total Chunks      : {len(chunks)}")
print(f"Average length      : {sum(lengths)//len(lengths)} chars")
print(f"Shortest section    : {min(lengths)} chars")
print(f"Longest section     : {max(lengths)} chars")
print(f"Sections over 4000  : {sum(1 for l in lengths if l > 4000)}")
print(f"Sections over 8000  : {sum(1 for l in lengths if l > 8000)}")
print(f"Sections over 16000 : {sum(1 for l in lengths if l > 16000)}")

Total Chunks      : 3955
Average length      : 2893 chars
Shortest section    : 72 chars
Longest section     : 4134 chars
Sections over 4000  : 1383
Sections over 8000  : 0
Sections over 16000 : 0


In [156]:
length1 = [len(c.page_content) for c in chunks]
length1.sort()
print(length1)

text = [c.page_content for c in chunks]
text.sort(key=len)
print(text[0])
print(f"Lengths: {len(text[0])}")

[72, 80, 80, 83, 84, 91, 92, 95, 104, 105, 112, 119, 120, 120, 122, 124, 124, 125, 128, 131, 131, 142, 143, 144, 146, 146, 150, 151, 157, 158, 158, 159, 161, 162, 165, 168, 170, 171, 172, 178, 178, 179, 180, 187, 193, 198, 200, 202, 202, 204, 208, 208, 212, 213, 217, 218, 218, 221, 221, 222, 222, 223, 223, 226, 227, 227, 227, 227, 227, 229, 230, 237, 237, 238, 240, 240, 241, 244, 244, 244, 247, 247, 247, 247, 252, 252, 253, 254, 255, 255, 256, 258, 258, 261, 262, 262, 263, 263, 267, 267, 268, 271, 272, 274, 275, 276, 277, 279, 280, 282, 283, 284, 286, 287, 287, 289, 289, 289, 290, 290, 291, 293, 295, 295, 297, 299, 299, 300, 300, 302, 304, 304, 306, 307, 307, 307, 308, 311, 313, 314, 314, 315, 315, 316, 318, 318, 320, 326, 326, 327, 328, 329, 329, 330, 331, 334, 335, 336, 337, 337, 338, 339, 341, 341, 347, 350, 351, 354, 354, 356, 356, 356, 357, 357, 358, 359, 360, 360, 361, 361, 364, 367, 368, 369, 374, 375, 378, 379, 379, 380, 380, 381, 384, 384, 384, 385, 385, 386, 388, 390, 392, 39